In [ ]:
# Practice Problems Day 6

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 6 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Reading:** Geron chapter 5, **Training and Visualizing a Decision Tree**, **Making Predictions**, **Regularization Hyperparameters**, and **Decision Trees Have High Variance**. Then chapter 6, **Voting Classifiers**, **Bagging and Pasting**, and **Random Forests**.

GOALS:

1. Grow a small decision tree and read it like a doctor would.
2. See why one big tree is a bad model and a forest of them is a good one.
3. Use out-of-bag scores and feature importance, and know what each one cannot tell you.
4. Choose a threshold on validation, open the test set once, and report what happened honestly.

**How much of this should you write yourself?** Every problem says. Problem 1 gives you the code with blanks. Problems 2 and 3 list the steps and you write the lines. Problem 4 is by hand. Problem 5 just tells you what to do. We use the same breast cancer data as the Day 5 homework, so the loading and splitting code is something you have already written once.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

---

## Problem 1: A two-question tree for cancer screening (fill in the blanks)

**What to copy, what to write:** Fill in the blanks marked `____`. Each one has a comment saying what goes there. Steps 1 and 2 are the same as the Day 5 homework, so they are done for you.

Remember the label: in our `y`, **1 means malignant** and 0 means benign.

**1a.** Fill in the blanks and run it. You should see a tree with seven boxes.

**1b.** What feature did the tree choose for its very first question?

**1c.** A new tumor has `mean concave points` of 0.03 and `worst area` of 1100. Trace it through the tree. Which leaf does it land in, what does the tree predict, and what is its probability of malignant? Show the division.

**1d.** Find the leaf with `value = [186, 4]`. The tree predicts benign for every tumor that lands there. What do the 4 mean for real patients? Which cell of the Day 5 confusion matrix are they?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Step 1: load the data, with 1 meaning MALIGNANT (same as the Day 5 homework)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series((data.target == 0).astype(int))

# Step 2: split the test set off first, then a validation set (same as the Day 5 homework)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)

# Step 3: a tree that asks at most two questions in a row
tree_small = DecisionTreeClassifier(
    max_depth=2,               # how many questions deep?
    random_state=42)
tree_small.fit(X_train, y_train)     # the training features and labels. No scaling needed!

# Step 4: draw it
plt.figure(figsize=(12, 6))
plot_tree(tree_small,
          feature_names=list(X.columns),
          class_names=["benign", "malignant"],       # class 0 first. In our y, 0 is benign and 1 is malignant
          filled=True,
          precision=3)
plt.show()

**1a.** The blanks: `2`, `X_train, y_train`, `"benign", "malignant"`.

**1b.** `mean concave points`, with the cutoff 0.051.

**1c.** 0.03 is at or below 0.051, so it goes left. `worst area` 1100 is above 949.85, so it goes right. It lands in the leaf with `value = [3, 6]`. The tree predicts **malignant**, with probability 6 / 9 = 0.667.

**1d.** Four training tumors in that leaf were malignant, and the tree calls every tumor there benign. Those are **false negatives**: patients with cancer that this tree would send home. They are the bottom-left cell of the Day 5 confusion matrix.

---

## Problem 2: One big tree against a forest (steps outlined)

**What to copy, what to write:** Nothing to copy. The steps are comments in the cell. Every one is a `LEARN TO WRITE THIS` cell in the Day 6 notes, so look there if you get stuck.

**2a.** Grow a tree with no `max_depth`. Print its training accuracy and its validation average precision.

**2b.** Print `np.unique()` of the big tree's validation probabilities. What values show up? Use that to explain why its average precision is worse than you might expect from its accuracy. (Hint: average precision is about **ranking** the tumors from most to least likely.)

**2c.** Now a random forest with 200 trees. Print its training accuracy and validation average precision.

**2d.** In the Day 5 homework, logistic regression got a validation average precision of **0.987** on these same tumors. Compare that to your forest. Which would you choose for a hospital, and why? There is more than one good answer.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, average_precision_score

# Step 1 (2a)
tree_big = DecisionTreeClassifier(random_state=42)
tree_big.fit(X_train, y_train)
print("depth:", tree_big.get_depth(), "  leaves:", tree_big.get_n_leaves())

# Step 2 (2a)
print("big tree training accuracy:", round(accuracy_score(y_train, tree_big.predict(X_train)), 3))
y_prob_tree = tree_big.predict_proba(X_valid)[:, 1]
print("big tree validation average precision:", round(average_precision_score(y_valid, y_prob_tree), 3))

# Step 3 (2b)
print("the probabilities it gives:", np.unique(y_prob_tree))

# Step 4 (2c)
forest = RandomForestClassifier(n_estimators=200, random_state=42)
forest.fit(X_train, y_train)
y_prob_forest = forest.predict_proba(X_valid)[:, 1]
print()
print("forest training accuracy:", round(accuracy_score(y_train, forest.predict(X_train)), 3))
print("forest validation average precision:", round(average_precision_score(y_valid, y_prob_forest), 3))

**2a.** The big tree is 5 deep with 12 leaves. Training accuracy 1.0, validation average precision 0.828.

**2b.** Only `0.` and `1.` show up. Every leaf of a fully grown tree is pure, so the tree is always completely sure. It cannot say "this one is more worrying than that one", so it cannot rank the tumors, and average precision is a score for ranking. The forest averages 200 trees, so its probabilities spread out between 0 and 1.

**2c.** Forest training accuracy 1.0, validation average precision 0.986.

**2d.** The forest (0.986) and logistic regression (0.987) are tied for practical purposes, and the difference is noise in 107 tumors. Either choice is fine if you give a reason. Good reasons: logistic regression is simpler and you can read its coefficients; the forest needs no scaling and handles odd shapes in the data. The important idea: **a forest is not automatically better.** On the wine data it won by a lot, here it does not.

---

## Problem 3: Out-of-bag and feature importance (steps outlined)

**What to copy, what to write:** You may copy the plotting code from the Day 6 notes. Write the forest and the scores yourself.

**3a.** Fit a forest with 500 trees and `oob_score=True` on **all** the training tumors (`X_train_full`). Print `oob_score_` next to the always-benign baseline. Is the forest doing much better than doing nothing?

**3b.** Compute the out-of-bag average precision. How does it compare to the forest's validation average precision from 2c?

**3c.** Print the 5 most important features from your Problem 2 forest.

**3d.** Three of the top five are `worst perimeter`, `worst area`, and `worst radius`. What do those three have in common? What does that mean for how you should read the importance chart?

In [ ]:
import pandas as pd

# Step 1 (3a)
forest_oob = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42)
forest_oob.fit(X_train_full, y_train_full)

# Step 2 (3a)
print("out-of-bag accuracy:", round(forest_oob.oob_score_, 3))
print("always-benign baseline:", round(1 - y_train_full.mean(), 3))

# Step 3 (3b)
y_prob_oob = forest_oob.oob_decision_function_[:, 1]
print("out-of-bag average precision:", round(average_precision_score(y_train_full, y_prob_oob), 3))

# Step 4 (3c)
importance = pd.Series(forest.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(5).round(3))
print()
print("correlation of worst perimeter and worst area:", round(X_train["worst perimeter"].corr(X_train["worst area"]), 3))

**3a.** Out-of-bag accuracy 0.955 against a baseline of 0.627. Yes, a lot better than doing nothing: the classes here are much less lopsided than the wine.

**3b.** Out-of-bag average precision 0.986, the same as validation. Two honest ways to get the same number.

**3c.** `worst perimeter` 0.142, `mean concave points` 0.121, `worst area` 0.114, `worst concave points` 0.095, `worst radius` 0.08.

**3d.** All three measure how **big** the tumor is. Radius, perimeter, and area of the same shape move together (perimeter and area have a correlation of 0.98 here). So the chart splits one piece of information, size, across three bars. Size matters more than any one of those bars suggests, and dropping one of the three would barely change the model.

---

## Problem 4: By hand, no code

**4a.** Five voters are each right 60 percent of the time, and their mistakes are independent. The majority is right when at least 3 of the 5 are right. What is the probability the majority is right? Show your arithmetic. (There are 10 ways to pick which 3 are right, 5 ways to pick which 4 are right, and 1 way for all 5.)

**4b.** A bootstrap sample of six rows, A through F, came back as `C, A, C, F, B, C`. Which rows are **out-of-bag** for the tree trained on this sample? What share of the original rows made it in?

**4c.** Your random forest has a training accuracy of 1.0. Your friend says it must be overfit and useless. Explain why that is wrong for a forest, even though it would be a real warning for a single tree.

**4d.** In plain bagging, every tree asks about the same strong feature first. Why is that bad for the crowd, and what does a random forest change to fix it?

In [ ]:
# checking 4a with code
from scipy.stats import binom
print(round(1 - binom.cdf(2, 5, 0.6), 4))

**4a.** 3 right: 10 x 0.6^3 x 0.4^2 = 10 x 0.216 x 0.16 = 0.3456. 4 right: 5 x 0.6^4 x 0.4 = 5 x 0.1296 x 0.4 = 0.2592. 5 right: 0.6^5 = 0.07776. Total **0.683**. The crowd beats the single voter's 0.6.

**4b.** D and E are out-of-bag. 4 of the 6 rows made it in (A, B, C, F), which is 0.667.

**4c.** Every tree is overfit to its own bootstrap sample, so the forest gets every training row right. But the trees overfit in different directions, so on new data their mistakes cancel in the average. A forest has to be judged on validation or out-of-bag data, never on training accuracy.

**4d.** If every tree asks the same first question, the trees look alike and make the same mistakes, and a vote of look-alikes cancels nothing. A random forest only lets each question choose from a random handful of features, so the trees are forced to differ.

---

## Problem 5: Screening threshold, then the test set once (just the task)

**What to copy, what to write:** All yours. The Day 5 and Day 6 notes have everything you need.

**5a.** Use the forest from Problem 2 and the **validation** set. Find the highest threshold that catches at least 39 of the 40 malignant validation tumors (recall of at least 0.975). Report the threshold, the precision, the recall, and the number of false alarms.

**5b.** Now, once, apply that threshold to the **test** set. Print the confusion matrix, precision, and recall.

**5c.** Validation promised a recall of at least 0.975. What did the test set give? Write two or three sentences for the doctor who will use this model: what the result means for patients, and why the test number is different from the validation number.

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

# 5a: walk down a list of thresholds on the validation set
thresholds = [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1]
for t in thresholds:
    y_pred_t = (y_prob_forest >= t).astype(int)
    r = recall_score(y_valid, y_pred_t)
    p = precision_score(y_valid, y_pred_t)
    false_alarms = ((y_pred_t == 1) & (y_valid.values == 0)).sum()
    print("threshold", t, "  recall", round(r, 3), "  precision", round(p, 3), "  false alarms", false_alarms)

In [ ]:
# the exact answer: the second lowest probability among the 40 malignant validation tumors
# is the highest threshold that still catches 39 of them
malignant_probs = np.sort(y_prob_forest[y_valid.values == 1])
print("lowest two malignant probabilities:", malignant_probs[:2])
print("highest threshold with recall >= 0.975:", malignant_probs[1])

In [ ]:
# 5b: the test set, once, with the threshold chosen on validation
chosen = 0.35     # the highest threshold from the list with recall >= 0.975 on validation
y_prob_test = forest.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= chosen).astype(int)
print(confusion_matrix(y_test, y_pred_test))
print("test precision:", round(precision_score(y_test, y_pred_test), 3))
print("test recall:   ", round(recall_score(y_test, y_pred_test), 3))

**5a.** From the list, **0.35** is the highest threshold with recall of at least 0.975 on validation: recall 0.975, precision 0.907, 4 false alarms. (0.4 and above catch only 38.) The exact highest threshold is **0.37**, the second lowest probability the forest gave a malignant validation tumor. The lowest is 0.055, one malignant tumor the forest was nearly sure was benign, and catching that one too would mean a threshold of 0.055. Either 0.35 or 0.37 is a good answer, and on this test set both give exactly the same result.

**5b.** On the test set at 0.35: `[[88, 2], [3, 50]]`. Precision 0.962, recall 0.943.

**5c.** Your answer will be different, here is one way to say it: "At the threshold we chose, the model caught 50 of the 53 cancers in patients it had never seen, and 3 were missed. We had expected to miss about 1 in 40, but that estimate came from only 40 cancers, so it was never very precise. If missing 3 in 53 is too many, the threshold has to come down, and that decision has to be made on new validation data, not by re-tuning on these patients."

The key ideas: say plainly that 3 cancers were missed, and that 40 cancers is a small sample to make a promise from.